# Módulo 05 - Programação Funcional

---

Python é uma linguagem **multi-paradigma**: o mesmo problema pode ser resolvido no estilo **imperativo** (laços e condicionais alterando variáveis passo a passo), **orientado a objetos** (classes que agrupam dados e comportamento) ou **funcional** (funções pequenas que transformam dados, sem alterar o que já existe). Neste módulo você vai conhecer o estilo funcional: primeiro aprende a criar funções anônimas com `lambda`, e depois combina essas funções com três ferramentas fundamentais — `map()`, `filter()` e `reduce()` — para transformar, selecionar e combinar dados de uma coleção sem escrever um laço `for` para cada etapa. O fio condutor será uma calculadora de juros compostos e o processamento de listas de e-mails e de números.

Curso: Ready To Deploy

Criado por: [Enzo Schitini](https://www.linkedin.com/in/enzoschitini)

---

## Tópicos

| **Tópico** | Descrição |
| --- | --- |
| 1. Função lambda | Funções anônimas de uma linha, boas práticas e funções de alta ordem, aplicadas a um calculador de juros. |
| 2. Função map | Transforma todos os itens de uma coleção, aplicada à extração de provedores de e-mail e a cenários de investimento. |
| 3. Função filter | Seleciona apenas os itens que atendem a uma condição, aplicada à filtragem de e-mails de um provedor. |
| 4. Função reduce | Combina todos os itens de uma coleção em um único valor, aplicada à busca do maior número de uma lista. |

---

## 1. Função lambda

O primeiro passo da programação funcional é conseguir criar funções pequenas e descartáveis sem todo o cerimonial de um `def`. É para isso que existe a função **`lambda`**.

### 1.1 Definição

Uma **função `lambda`** é uma função **anônima** (sem nome), escrita em uma única linha, contendo apenas **uma expressão** — não pode ter laços, `print()` ou múltiplas instruções. Sua sintaxe é:

```python
variavel = lambda parametros: expressao
```

A `expressao` é automaticamente o valor de retorno, sem precisar da palavra `return`.

**Exemplo:** extraindo o provedor de um e-mail.

In [1]:
extrair_provedor_email = lambda email: email.split(sep='@')[-1]

In [2]:
email = 'andre.perez@gmail.com'
print(email)

provedor_email = extrair_provedor_email(email)
print(provedor_email)

andre.perez@gmail.com
gmail.com


### 1.2 Boas práticas

Como uma `lambda` só pode conter uma expressão, é comum usar o **operador ternário** (`valor_se_verdadeiro if condicao else valor_se_falso`) para embutir uma decisão simples.

**Exemplo:** verificando se um número é par.

In [3]:
# Funciona, mas o if/else é redundante: a comparação já é um booleano
numero_e_par_verboso = lambda numero: True if numero % 2 == 0 else False

# Forma direta e mais idiomática: basta retornar a própria comparação
numero_e_par = lambda numero: numero % 2 == 0

print(numero_e_par_verboso(4))
print(numero_e_par(4))

True
True


In [4]:
for numero in range(10):
    if numero_e_par(numero):
        print(f'O número {numero} é par!')

O número 0 é par!
O número 2 é par!
O número 4 é par!
O número 6 é par!
O número 8 é par!


> ⚠️ **Atenção:** por ser limitada a uma expressão, `lambda` não substitui `def` em toda situação. Use `lambda` para lógicas curtas e, principalmente, passadas diretamente como argumento de outra função (como veremos em `map`, `filter` e `reduce`). Para uma lógica com várias linhas ou que precisa de nome próprio no código, prefira `def` — a PEP 8 inclusive recomenda não atribuir uma `lambda` a uma variável quando um `def` deixaria o código mais claro.

### 1.3 Funções de alta ordem

Uma **função de alta ordem** é uma função que **recebe outra função como parâmetro** ou que **retorna uma função**. Combinar isso com `lambda` é muito poderoso: uma função pode "fabricar" outras funções sob medida.

**Exemplo:** uma calculadora de retorno de investimento, em que a taxa de juros já vem embutida na função gerada.

In [5]:
def criar_calculadora_retorno(taxa_juros: float):
    # A lambda retornada "lembra" do valor de taxa_juros mesmo depois
    # que criar_calculadora_retorno() já terminou de executar — isso é
    # chamado de closure.
    return lambda investimento: investimento * (1 + taxa_juros)

In [6]:
retorno_5_porcento = criar_calculadora_retorno(taxa_juros=0.05)
retorno_10_porcento = criar_calculadora_retorno(taxa_juros=0.10)

In [7]:
# Cada calculadora já "sabe" sua própria taxa de juros
valor_final = retorno_5_porcento(investimento=1000)
print(round(valor_final, 2))

valor_final = retorno_10_porcento(investimento=1000)
print(round(valor_final, 2))

1050.0
1100.0


Como as calculadoras retornam um valor, podemos **encadeá-las em um laço** para simular juros compostos ao longo de vários anos:

In [8]:
anos = 10
valor_final = 1000

for _ in range(anos):
    valor_final = retorno_5_porcento(investimento=valor_final)

print(round(valor_final, 2))

1628.89


In [9]:
anos = 10
valor_final = 1000

for _ in range(anos):
    valor_final = retorno_10_porcento(investimento=valor_final)

print(round(valor_final, 2))

2593.74


---

## 2. Função map

Com `lambda` já dominado, podemos combiná-la com funções que operam sobre **coleções inteiras** de uma só vez. A primeira delas é `map()`, que transforma cada item de uma coleção.

### 2.1 Definição

A função `map()` aplica uma função a **todos** os elementos de uma coleção (`list`, `dict` etc.) e retorna **todos** os elementos já transformados:

```python
variavel = map(funcao, colecao)
```

In [10]:
numeros = [1, 2, 3]

numeros_ao_cubo = map(lambda num: num ** 3, numeros)
print(numeros_ao_cubo)

> ⚠️ **Atenção:** `map()` não retorna uma `list`, e sim um **iterador** — por isso `print()` mostra apenas o endereço do objeto na memória, e não os valores. Para ver ou usar os resultados, converta com `list()`. Além disso, um iterador só pode ser **percorrido uma vez**: depois de convertido (ou usado em um laço), ele fica "vazio".

In [11]:
print(list(numeros_ao_cubo))

[1, 8, 27]


### 2.2 Substituindo laços por map

**Exemplo:** extraindo o provedor de vários e-mails. Primeiro, a forma imperativa, com um laço `for`:

In [12]:
emails = ['andre.perez@gmail.com', 'andre.perez@live.com', 'andre.perez@yahoo.com']
extrair_provedor_email = lambda email: email.split(sep='@')[-1]

In [13]:
# Forma imperativa: construímos a lista manualmente, item a item
provedores = []
for email in emails:
    provedor = extrair_provedor_email(email)
    provedores.append(provedor)

print(provedores)

['gmail.com', 'live.com', 'yahoo.com']


Agora, a mesma tarefa no estilo funcional, com `map()` — sem laço, sem lista vazia para preencher:

In [14]:
provedores = list(map(extrair_provedor_email, emails))
print(provedores)

['gmail.com', 'live.com', 'yahoo.com']


> 💡 **Dica:** quando a função é usada em um único lugar, não há necessidade de nomeá-la — podemos passar a `lambda` diretamente como argumento de `map()`.

In [15]:
provedores = list(map(lambda email: email.split(sep='@')[-1], emails))
print(provedores)

['gmail.com', 'live.com', 'yahoo.com']


### 2.3 map com múltiplos parâmetros

`map()` também aceita **mais de uma coleção**: nesse caso, a função passada precisa receber um parâmetro para cada coleção, e os itens são combinados posição a posição (o 1º item de cada lista, depois o 2º, e assim por diante).

**Exemplo:** calculando o retorno de vários cenários de investimento de uma só vez.

In [16]:
def calcular_retorno_investimento(valor_inicial: float, taxa_juros: float, anos: int) -> float:
    valor_final = valor_inicial
    for _ in range(anos):
        valor_final = valor_final * (1 + taxa_juros)
    return round(valor_final, 2)

In [17]:
valores_iniciais = [1000, 1000, 1000]
taxas_juros = [0.05, 0.10, 0.15]
anos = [10, 10, 10]

cenarios = list(map(calcular_retorno_investimento, valores_iniciais, taxas_juros, anos))
print(cenarios)

[1628.89, 2593.74, 4045.56]


---

## 3. Função filter

Enquanto `map()` transforma todos os itens, às vezes queremos apenas **selecionar** alguns deles. Para isso existe `filter()`.

### 3.1 Definição

A função `filter()` aplica uma função **lógica** (que retorna `True` ou `False`) a todos os elementos de uma coleção, e retorna **apenas** os elementos para os quais o resultado foi `True`:

```python
variavel = filter(funcao_logica, colecao)
```

Assim como `map()`, o retorno de `filter()` também é um **iterador**.

In [18]:
numeros = [1, 2, 3, 4, 5, 6]

numeros_pares = filter(lambda num: num % 2 == 0, numeros)
print(list(numeros_pares))

[2, 4, 6]


### 3.2 Substituindo laços por filter

**Exemplo:** selecionando apenas os e-mails de um provedor específico. De novo, comparando a forma imperativa com a funcional.

In [19]:
emails = ['andre.perez@gmail.com', 'andre.perez@live.com', 'andre.perez@yahoo.com']
eh_do_gmail = lambda email: 'gmail' in email

In [20]:
# Forma imperativa
emails_gmail = []
for email in emails:
    if eh_do_gmail(email):
        emails_gmail.append(email)

print(emails_gmail)

['andre.perez@gmail.com']


In [21]:
# Forma funcional, equivalente
emails_gmail = list(filter(eh_do_gmail, emails))
print(emails_gmail)

['andre.perez@gmail.com']


> 💡 **Dica:** o critério do filtro raramente é reaproveitado em outro lugar do código — por isso, é muito comum escrevê-lo diretamente como uma `lambda`, sem nomeá-lo antes.

In [22]:
emails_gmail = list(filter(lambda email: 'gmail' in email, emails))
print(emails_gmail)

['andre.perez@gmail.com']


---

## 4. Função reduce

`map()` transforma e `filter()` seleciona, mas ambos ainda retornam uma **coleção**. Quando o objetivo é condensar tudo em um **único valor** — uma soma, o maior item, uma concatenação — usamos `reduce()`.

### 4.1 Definição

A função `reduce()` aplica uma função a todos os elementos de uma coleção, **dois a dois**, acumulando o resultado, até restar **um único valor**:

```python
variavel = reduce(funcao, colecao)
```

> ⚠️ **Atenção:** diferente de `map()` e `filter()`, `reduce()` não é uma função nativa do Python — desde o Python 3 ela precisa ser importada do módulo `functools`.

In [23]:
from functools import reduce

numeros = [1, 2, 3, 4]

soma = reduce(lambda acumulado, atual: acumulado + atual, numeros)
print(soma)

10


### 4.2 Funções de alta ordem com reduce

**Exemplo:** encontrando o maior número de uma lista, sem usar a função pronta `max()`.

In [24]:
def maior_entre(primeiro: int, segundo: int) -> int:
    return primeiro if primeiro >= segundo else segundo

print(maior_entre(11, 4))

11


In [25]:
from random import random

# Lista com 100 números inteiros aleatórios entre 0 e 100
# (list comprehension: veremos essa sintaxe em detalhe em um módulo futuro)
numeros = [round(100 * random()) for _ in range(100)]
print(numeros[:10], '...')  # mostrando só os 10 primeiros

[47, 21, 4, 15, 42, 56, 60, 35, 71, 84] ...


In [26]:
# reduce aplica maior_entre() aos dois primeiros números, depois ao
# resultado com o terceiro, depois ao resultado com o quarto, e assim
# por diante, até sobrar só o maior de todos
maior_numero = reduce(maior_entre, numeros)
print(maior_numero)

100


In [27]:
# A mesma lógica, com a função escrita como lambda
maior_numero = reduce(lambda primeiro, segundo: primeiro if primeiro >= segundo else segundo, numeros)
print(maior_numero)

100


> 💡 **Dica:** para encontrar o maior valor de uma coleção, o Python já tem a função nativa `max()` — muito mais direta que `reduce()` para esse caso específico. O exemplo acima tem fins didáticos: `reduce()` realmente se destaca quando a lógica de combinação é **personalizada** e não existe uma função pronta para ela.

### 4.3 Combinando map, filter e reduce

Como `map()`, `filter()` e `reduce()` seguem o mesmo padrão (função + coleção), eles podem ser **encadeados**: a saída de um vira a entrada do próximo.

**Exemplo:** somando o quadrado apenas dos números originalmente ímpares.

In [28]:
from random import random

numeros = [round(100 * random()) for _ in range(100)]
print(numeros[:10], '...')

[50, 3, 65, 46, 43, 99, 40, 69, 84, 98] ...


**Passo 1:** elevar cada número ao quadrado (`map`).

In [29]:
numeros_ao_quadrado = map(lambda numero: numero ** 2, numeros)

**Passo 2:** manter apenas os quadrados de números ímpares — o quadrado de um número ímpar também é ímpar (`filter`).

In [30]:
quadrados_impares = filter(lambda numero: numero % 2 != 0, numeros_ao_quadrado)

**Passo 3:** somar tudo em um único valor (`reduce`).

In [31]:
soma_quadrados_impares = reduce(lambda acumulado, atual: acumulado + atual, quadrados_impares)
print(soma_quadrados_impares)

190475


> ⚠️ **Atenção:** `map` e `filter` retornam iteradores que só podem ser percorridos **uma vez**. Por isso, o exemplo abaixo recomeça a partir da lista `numeros` original — reaproveitar `numeros_ao_quadrado` ou `quadrados_impares` aqui não funcionaria, pois ambos já foram consumidos no passo 3 acima.

In [32]:
soma_quadrados_impares = reduce(
    lambda acumulado, atual: acumulado + atual,
    filter(
        lambda numero: numero % 2 != 0,
        map(lambda numero: numero ** 2, numeros),
    ),
)
print(soma_quadrados_impares)

190475


> 💡 Embora o encadeamento acima funcione em uma única linha, ele fica difícil de ler. No dia a dia, prefira o formato passo a passo (como fizemos antes) sempre que a linha ficar longa demais — código funcional não precisa ser compacto para ser bom.

---

## Resumo do Módulo

| Ferramenta | O que faz | Retorna | Exemplo |
| --- | --- | --- | --- |
| `lambda` | Cria uma função anônima de uma expressão só | Função | `lambda x: x ** 2` |
| `map(f, colecao)` | Transforma cada item da coleção | Iterador | `map(lambda x: x * 2, numeros)` |
| `filter(f, colecao)` | Mantém apenas os itens em que `f` retorna `True` | Iterador | `filter(lambda x: x > 0, numeros)` |
| `reduce(f, colecao)` | Combina todos os itens em um único valor (requer `from functools import reduce`) | Valor único | `reduce(lambda a, b: a + b, numeros)` |

Funções nativas e de alta ordem vistas neste módulo: `lambda`, `map()`, `filter()`, `reduce()` (de `functools`), `list()`, `max()`.